# PreProcessing Data

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

resolution = "square_008um"
h5_path = "binned_outputs/"+ resolution +"/filtered_feature_bc_matrix.h5"
spatial_path = "binned_outputs/"+ resolution +"/spatial/tissue_positions.parquet" 

print("Loading data...")
adata = sc.read_10x_h5(h5_path)
if spatial_path.endswith('.parquet'):
    positions = pd.read_parquet(spatial_path)
    positions.set_index('barcode', inplace=True)
else:
    positions = pd.read_csv(spatial_path, index_col=0, header=None)

# Visium HD stores array coordinates in specific columns. 
positions = positions.loc[adata.obs_names] 
adata.obsm['spatial'] = positions[['pxl_col_in_fullres', 'pxl_row_in_fullres']].values

print("Exporting spatial coordinates...")
np.savetxt("visium_spatial_coords.csv", adata.obsm['spatial'], delimiter=",", fmt="%.3f")

# adata.X is a CSR matrix. We only want the column indices of the non-zero elements!
print("Exporting sparse gene indices...")
X_csr = adata.X.tocsr()
with open("visium_sparse_genes.txt", "w") as f:
    for i in range(X_csr.shape[0]):
        
        start = X_csr.indptr[i]
        end = X_csr.indptr[i+1]
        
        # Extract the column indices (the gene IDs) that have > 0 counts
        gene_indices = X_csr.indices[start:end]
        
        f.write(",".join(map(str, gene_indices)) + "\n")
print("Data successfully preprocessed")

Loading data...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/anndata/_core/anndata.py:1880: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Exporting spatial coordinates...
Exporting sparse gene indices...
Data successfully preprocessed
